In [18]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from IPython import display
import mediapipe as mp
import numpy as np
import os



In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


Load Gender Model

In [20]:
# -- Load Gender Model
gender_weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
gender_model = torchvision.models.efficientnet_b0(weights=gender_weights).to(device)


In [21]:
for param in gender_model.parameters():
    param.requires_grad = True

In [22]:
gender_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(1280, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 1)

).to(device)

Load Age Model

In [23]:
# -- Load Age Model
age_weights = torchvision.models.VGG19_Weights.DEFAULT
age_model = torchvision.models.vgg19( weights=age_weights)

In [24]:
# Freeze all parameters in the feature layers
for param in age_model.features.parameters():
    param.requires_grad = False

# Unfreeze only the last 12 layers of the feature layers
for param in age_model.features[-20:].parameters():
    param.requires_grad = True



In [25]:
age_model.classifier = nn.Sequential(
    
    nn.Sequential(
        nn.Linear(512 * 7 * 7, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU(),
        nn.Dropout(0.5),  
        nn.Linear(4096, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU()
    ),

    
    nn.Sequential(
        nn.Linear(4096, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(),
        nn.Dropout(0.4),  
        nn.Linear(2048, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(),
        
    ),

    
    nn.Sequential(
        nn.Linear(2048, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(1024, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        
    ),

    
    nn.Sequential(
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.2),  
        nn.Linear(512, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        
    ),

    
    nn.Sequential(
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.1),  
        nn.Linear(256, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        
    ),

    
    nn.Linear(256, 1)
).to(device)

Load Emotion Model

In [26]:
# -- Load Emotion Model
emotion_weights = torchvision.models.VGG19_Weights.DEFAULT
emotion_model = torchvision.models.vgg19( weights=emotion_weights)

In [27]:
# Freeze all parameters in the feature layers
for param in emotion_model.features.parameters():
    param.requires_grad = False

for param in emotion_model.features[-20:].parameters():
    param.requires_grad = True    


# Modify the classifier
emotion_model.classifier = nn.Sequential(
    
    nn.Sequential(
        nn.Linear(512 * 7 * 7, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(4096, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU()
    ),

    
    nn.Sequential(
        nn.Linear(4096, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(2048, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(),
        
    ),

    
    nn.Sequential(
        nn.Linear(2048, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(1024, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        
    ),

    
    nn.Sequential(
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        
    ),

    
    nn.Sequential(
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        
    ),

    
    nn.Linear(256, 7)  # 7 emotion classes
).to(device)




In [28]:
gender_model.load_state_dict(torch.load(os.path.join(os.path.dirname(os.path.abspath("FacePredictor_app.ipynb")), "fine_tuned_gender_model.pth"), map_location=device))
gender_model.to(device).eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [29]:
age_model.load_state_dict(torch.load(os.path.join(os.path.dirname(os.path.abspath("FacePredictor_app.ipynb")), "fine_tuned_age_model.pth"), map_location=device))
age_model.to(device).eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [30]:
emotion_model.load_state_dict(torch.load(os.path.join(os.path.dirname(os.path.abspath("FacePredictor_app.ipynb")), "emotion_model_fine_tuned.pth"), map_location=device))
emotion_model.to(device).eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

# Data Transformation

In [31]:
# Data transformation for gender model
gender_infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [32]:
# Data transformation for age model
age_infer_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])



In [33]:
# Data transformation for emotion model
emotion_infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# Prediction

In [34]:
def predict_and_display(frame, face_box, gender_model, age_model, emotion_model, all_face_boxes=None):
    # Extract face region from frame
    x, y, w, h = face_box
    face = frame[y:y+h, x:x+w]

    # Convert to PIL image for model input
    face_pil = Image.fromarray(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))

    # Preprocess face for each model
    gender_tensor = gender_infer_transform(face_pil).unsqueeze(0).to(device)
    age_tensor = age_infer_transform(face_pil).unsqueeze(0).to(device)
    emotion_tensor = emotion_infer_transform(face_pil).unsqueeze(0).to(device)

    # Run predictions
    with torch.no_grad():
        gender_output = gender_model(gender_tensor)
        age_output = age_model(age_tensor)
        emotion_output = emotion_model(emotion_tensor)
        
        # Process gender prediction
        gender_pred = torch.round(torch.sigmoid(gender_output))
        gender_index = int(gender_pred.item())
        gender_labels = ['Male', 'Female']
        gender_text = gender_labels[gender_index]
        
        # Process emotion prediction
        age_text = int(age_output.item())
        
        # Process emotion prediction
        emotion_index = torch.argmax(emotion_output, dim=1).item()
        emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
        emotion_text = emotion_labels[emotion_index]

    # Draw bounding box and results
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    labels = [
        f"Gender: {gender_text}",
        f"Age: {age_text}",
        f"Emotion: {emotion_text}"
    ]
    
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.65
    thickness = 2
    line_spacing = 8
    padding = 10

    # Calculate dimensions of the text block
    line_heights = []
    line_widths = []
    for line in labels:
        (lw, lh), _ = cv2.getTextSize(line, font, font_scale, thickness)
        line_heights.append(lh)
        line_widths.append(lw)
    
    total_text_height = sum(line_heights) + line_spacing * (len(labels) - 1)
    space_needed = total_text_height + padding

    # Decide position: 'top' or 'bottom'
    top_fits = (y - space_needed >= 0)
    bottom_fits = (y + h + space_needed <= frame.shape[0])
    
    top_overlaps = False
    bottom_overlaps = False
    
    if all_face_boxes and len(all_face_boxes) > 1:
        top_ymin, top_ymax = y - space_needed, y
        top_xmin, top_xmax = x - 5, x + w + 5
        
        bottom_ymin, bottom_ymax = y + h, y + h + space_needed
        bottom_xmin, bottom_xmax = x - 5, x + w + 5
        
        for ob in all_face_boxes:
            if ob == face_box:
                continue
            ox, oy, ow, oh = ob
            # Check overlap with top area
            if (ox < top_xmax and ox + ow > top_xmin and 
                oy < top_ymax and oy + oh > top_ymin):
                top_overlaps = True
            # Check overlap with bottom area
            if (ox < bottom_xmax and ox + ow > bottom_xmin and 
                oy < bottom_ymax and oy + oh > bottom_ymin):
                bottom_overlaps = True

    # Decide placement based on space and overlap
    if top_fits and not top_overlaps:
        position = 'top'
    elif bottom_fits and not bottom_overlaps:
        position = 'bottom'
    elif top_fits and bottom_fits:
        position = 'top'
    elif top_fits:
        position = 'top'
    elif bottom_fits:
        position = 'bottom'
    else:
        position = 'top'

    # Draw the text block
    if position == 'top':
        current_y = y - padding - total_text_height
    else:
        current_y = y + h + padding

    for i, line in enumerate(labels):
        lh = line_heights[i]
        lw = line_widths[i]
        
        # Left-align with face box, but keep within frame boundaries
        text_x = x
        if text_x + lw > frame.shape[1] - 10:
            text_x = max(10, frame.shape[1] - lw - 10)
            
        cv2.putText(frame, line, (text_x, current_y + lh), 
                    font, font_scale, (255, 0, 0), thickness)
        current_y += lh + line_spacing

    return frame


def main():
    # Initialize MediaPipe face detector
    mp_face_detection = mp.solutions.face_detection
    mp_drawing = mp.solutions.drawing_utils

    # Open webcam
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Could not open webcam.")
        return

    # Start face detection
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.6) as face_detection:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Convert frame to RGB for MediaPipe
            height, width, _ = frame.shape
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_detection.process(frame_rgb)

            # If faces detected, process each
            if results.detections:
                face_boxes = []
                for detection in results.detections:
                    bbox = detection.location_data.relative_bounding_box
                    x = int(bbox.xmin * width)
                    y = int(bbox.ymin * height)
                    w = int(bbox.width * width)
                    h = int(bbox.height * height)

                    # Clamp values to frame size
                    x, y = max(0, x), max(0, y)
                    w, h = min(w, width - x), min(h, height - y)
                    face_boxes.append((x, y, w, h))

                for face_box in face_boxes:
                    frame = predict_and_display(frame, face_box,
                                                gender_model, age_model, emotion_model, face_boxes)
                    
            # Show the output
            cv2.imshow('Webcam Face Analysis', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == '__main__':
    main()
